# Normalized Market Data Viewer

Inspect the normalized Parquet files produced by `modl-normalize`.

Defaults:
- root: `/mnt/burner-archive/ws_normalized`
- date: `2026-06-30`

Override with `MODL_WS_NORMALIZED_DIR` or `MODL_VIEW_DATE` before launching Jupyter.

In [1]:
from __future__ import annotations

import os
from datetime import datetime
from pathlib import Path

import pandas as pd
import polars as pl

ROOT = Path(os.environ.get("MODL_WS_NORMALIZED_DIR", "/mnt/burner-archive/ws_normalized")).expanduser()
DATE = os.environ.get("MODL_VIEW_DATE", "2026-06-30")
DATE_TAG = datetime.strptime(DATE, "%Y-%m-%d").strftime("%y-%m-%d")

pd.set_option("display.max_columns", 160)
pd.set_option("display.max_colwidth", 180)
pl.Config.set_tbl_cols(160)
pl.Config.set_tbl_rows(20)

ROOT, DATE, DATE_TAG

(PosixPath('/mnt/burner-archive/ws_normalized'), '2026-06-30', '26-06-30')

In [2]:
def normalized_parts(path: Path, root: Path = ROOT) -> dict[str, str]:
    rel = path.relative_to(root)
    parts = rel.parts
    if len(parts) < 4:
        raise ValueError(f"unexpected normalized path shape: {path}")
    return {
        "exchange": parts[0],
        "symbol": parts[1],
        "dataset": parts[2],
        "file": parts[-1],
    }


def discover_files(root: Path = ROOT, date_tag: str = DATE_TAG) -> pd.DataFrame:
    rows = []
    for path in sorted(root.rglob(f"*_{date_tag}.parquet")):
        parts = normalized_parts(path, root)
        stat = path.stat()
        rows.append(
            {
                **parts,
                "key": f"{parts['exchange']}/{parts['symbol']}/{parts['dataset']}",
                "size_mb": round(stat.st_size / 1_000_000, 3),
                "path": str(path),
            }
        )
    return pd.DataFrame(rows)


files = discover_files()
files

,exchange,symbol,dataset,file,key,size_mb,path
0,bitfinex,tbtcusd,book_l25,tbtcusd_book_l25_26-06-30.parquet,bitfinex/tbtcusd/book_l25,1.543,/mnt/burner-archive/ws_normalized/bitfinex/tbtcusd/book_l25/tbtcusd_book_l25_26-06-30.parquet
1,bitfinex,tbtcusd,trades,tbtcusd_trades_26-06-30.parquet,bitfinex/tbtcusd/trades,0.075,/mnt/burner-archive/ws_normalized/bitfinex/tbtcusd/trades/tbtcusd_trades_26-06-30.parquet
2,deribit,btc,incremental_ticker,btc_incremental_ticker_26-06-30.parquet,deribit/btc/incremental_ticker,27.255,/mnt/burner-archive/ws_normalized/deribit/btc/incremental_ticker/btc_incremental_ticker_26-06-30.parquet
3,deribit,btc,instrument_state,btc_instrument_state_26-06-30.parquet,deribit/btc/instrument_state,0.063,/mnt/burner-archive/ws_normalized/deribit/btc/instrument_state/btc_instrument_state_26-06-30.parquet
4,deribit,btc,instruments,btc_instruments_26-06-30.parquet,deribit/btc/instruments,0.015,/mnt/burner-archive/ws_normalized/deribit/btc/instruments/btc_instruments_26-06-30.parquet
5,deribit,btc,trades,btc_trades_26-06-30.parquet,deribit/btc/trades,0.141,/mnt/burner-archive/ws_normalized/deribit/btc/trades/btc_trades_26-06-30.parquet
6,hibachi,btc_usdt-p,funding,btc_usdt-p_funding_26-06-30.parquet,hibachi/btc_usdt-p/funding,0.162,/mnt/burner-archive/ws_normalized/hibachi/btc_usdt-p/funding/btc_usdt-p_funding_26-06-30.parquet
7,hibachi,btc_usdt-p,orderbook,btc_usdt-p_orderbook_26-06-30.parquet,hibachi/btc_usdt-p/orderbook,1.151,/mnt/burner-archive/ws_normalized/hibachi/btc_usdt-p/orderbook/btc_usdt-p_orderbook_26-06-30.parquet
8,hibachi,btc_usdt-p,prices,btc_usdt-p_prices_26-06-30.parquet,hibachi/btc_usdt-p/prices,0.377,/mnt/burner-archive/ws_normalized/hibachi/btc_usdt-p/prices/btc_usdt-p_prices_26-06-30.parquet
9,hibachi,btc_usdt-p,quotes,btc_usdt-p_quotes_26-06-30.parquet,hibachi/btc_usdt-p/quotes,0.233,/mnt/burner-archive/ws_normalized/hibachi/btc_usdt-p/quotes/btc_usdt-p_quotes_26-06-30.parquet


In [3]:
if files.empty:
    raise FileNotFoundError(f"No normalized Parquet files found under {ROOT} for {DATE}")

DATASETS = dict(zip(files["key"], files["path"]))


def scan_dataset(key: str) -> pl.LazyFrame:
    return pl.scan_parquet(DATASETS[key])


def load_dataset(key: str, limit: int | None = None) -> pl.DataFrame:
    lf = scan_dataset(key)
    if limit is not None:
        lf = lf.head(limit)
    return lf.collect()


def row_count(key: str) -> int:
    return scan_dataset(key).select(pl.len().alias("rows")).collect().item()


summary = files[["exchange", "symbol", "dataset", "key", "size_mb", "path"]].copy()
summary["rows"] = [row_count(key) for key in summary["key"]]
summary.sort_values(["exchange", "symbol", "dataset"]).reset_index(drop=True)

,exchange,symbol,dataset,key,size_mb,path,rows
0,bitfinex,tbtcusd,book_l25,bitfinex/tbtcusd/book_l25,1.543,/mnt/burner-archive/ws_normalized/bitfinex/tbtcusd/book_l25/tbtcusd_book_l25_26-06-30.parquet,158343
1,bitfinex,tbtcusd,trades,bitfinex/tbtcusd/trades,0.075,/mnt/burner-archive/ws_normalized/bitfinex/tbtcusd/trades/tbtcusd_trades_26-06-30.parquet,4458
2,deribit,btc,incremental_ticker,deribit/btc/incremental_ticker,27.255,/mnt/burner-archive/ws_normalized/deribit/btc/incremental_ticker/btc_incremental_ticker_26-06-30.parquet,888043
3,deribit,btc,instrument_state,deribit/btc/instrument_state,0.063,/mnt/burner-archive/ws_normalized/deribit/btc/instrument_state/btc_instrument_state_26-06-30.parquet,4504
4,deribit,btc,instruments,deribit/btc/instruments,0.015,/mnt/burner-archive/ws_normalized/deribit/btc/instruments/btc_instruments_26-06-30.parquet,934
5,deribit,btc,trades,deribit/btc/trades,0.141,/mnt/burner-archive/ws_normalized/deribit/btc/trades/btc_trades_26-06-30.parquet,5252
6,hibachi,btc_usdt-p,funding,hibachi/btc_usdt-p/funding,0.162,/mnt/burner-archive/ws_normalized/hibachi/btc_usdt-p/funding/btc_usdt-p_funding_26-06-30.parquet,17170
7,hibachi,btc_usdt-p,orderbook,hibachi/btc_usdt-p/orderbook,1.151,/mnt/burner-archive/ws_normalized/hibachi/btc_usdt-p/orderbook/btc_usdt-p_orderbook_26-06-30.parquet,136117
8,hibachi,btc_usdt-p,prices,hibachi/btc_usdt-p/prices,0.377,/mnt/burner-archive/ws_normalized/hibachi/btc_usdt-p/prices/btc_usdt-p_prices_26-06-30.parquet,34340
9,hibachi,btc_usdt-p,quotes,hibachi/btc_usdt-p/quotes,0.233,/mnt/burner-archive/ws_normalized/hibachi/btc_usdt-p/quotes/btc_usdt-p_quotes_26-06-30.parquet,17170


## Pick A Dataset

Set `key` to any value from the summary table, then rerun the cells below.

In [4]:
sorted(DATASETS)

['bitfinex/tbtcusd/book_l25',
 'bitfinex/tbtcusd/trades',
 'deribit/btc/incremental_ticker',
 'deribit/btc/instrument_state',
 'deribit/btc/instruments',
 'deribit/btc/trades',
 'hibachi/btc_usdt-p/funding',
 'hibachi/btc_usdt-p/orderbook',
 'hibachi/btc_usdt-p/prices',
 'hibachi/btc_usdt-p/quotes',
 'hibachi/btc_usdt-p/trades',
 'hyperliquid/ubtc_usdc/book',
 'hyperliquid/ubtc_usdc/control',
 'hyperliquid/ubtc_usdc/trades']

In [14]:
key = "deribit/btc/instruments"
df = load_dataset(key)
df.head(20)

exchange,symbol,instrument_name,kind,state,base_currency,quote_currency,settlement_currency,counter_currency,instrument_type,future_type,settlement_period,option_type,price_index,strike,contract_size,tick_size,expiration_timestamp,creation_timestamp,instrument_id,is_active
str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,i64,i64,i64,bool
"""deribit""","""btc""","""BTC-10JUL26""","""future""","""open""","""BTC""","""USD""","""BTC""","""USD""","""reversed""","""reversed""","""week""",null,"""btc_usd""",null,"""10.0""","""2.5""",1783670400000,1781769612000,648483,true
"""deribit""","""btc""","""BTC-10JUL26-48000-C""","""option""","""open""","""BTC""","""BTC""","""BTC""","""USD""","""reversed""",null,"""week""","""call""","""btc_usd""","""4.8e+4""","""1.0""","""0.0001""",1783670400000,1782360775000,651575,true
"""deribit""","""btc""","""BTC-10JUL26-48000-P""","""option""","""open""","""BTC""","""BTC""","""BTC""","""USD""","""reversed""",null,"""week""","""put""","""btc_usd""","""4.8e+4""","""1.0""","""0.0001""",1783670400000,1782360775000,651576,true
"""deribit""","""btc""","""BTC-10JUL26-50000-C""","""option""","""open""","""BTC""","""BTC""","""BTC""","""USD""","""reversed""",null,"""week""","""call""","""btc_usd""","""5.0e+4""","""1.0""","""0.0001""",1783670400000,1782197502000,650734,true
"""deribit""","""btc""","""BTC-10JUL26-50000-P""","""option""","""open""","""BTC""","""BTC""","""BTC""","""USD""","""reversed""",null,"""week""","""put""","""btc_usd""","""5.0e+4""","""1.0""","""0.0001""",1783670400000,1782197502000,650735,true
"""deribit""","""btc""","""BTC-10JUL26-52000-C""","""option""","""open""","""BTC""","""BTC""","""BTC""","""USD""","""reversed""",null,"""week""","""call""","""btc_usd""","""5.2e+4""","""1.0""","""0.0001""",1783670400000,1781811758000,649129,true
"""deribit""","""btc""","""BTC-10JUL26-52000-P""","""option""","""open""","""BTC""","""BTC""","""BTC""","""USD""","""reversed""",null,"""week""","""put""","""btc_usd""","""5.2e+4""","""1.0""","""0.0001""",1783670400000,1781811758000,649130,true
"""deribit""","""btc""","""BTC-10JUL26-54000-C""","""option""","""open""","""BTC""","""BTC""","""BTC""","""USD""","""reversed""",null,"""week""","""call""","""btc_usd""","""5.4e+4""","""1.0""","""0.0001""",1783670400000,1781811721000,649127,true
"""deribit""","""btc""","""BTC-10JUL26-54000-P""","""option""","""open""","""BTC""","""BTC""","""BTC""","""USD""","""reversed""",null,"""week""","""put""","""btc_usd""","""5.4e+4""","""1.0""","""0.0001""",1783670400000,1781811721000,649128,true


In [16]:
df_stats = df.describe()
df_stats

statistic,exchange,symbol,instrument_name,kind,state,base_currency,quote_currency,settlement_currency,counter_currency,instrument_type,future_type,settlement_period,option_type,price_index,strike,contract_size,tick_size,expiration_timestamp,creation_timestamp,instrument_id,is_active
str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,f64,f64,f64,f64
"""count""","""934""","""934""","""934""","""934""","""934""","""934""","""934""","""934""","""934""","""934""","""14""","""934""","""920""","""934""","""920""","""934""","""934""",934.0,934.0,934.0,934.0
"""null_count""","""0""","""0""","""0""","""0""","""0""","""0""","""0""","""0""","""0""","""0""","""920""","""0""","""14""","""0""","""14""","""0""","""0""",0.0,0.0,0.0,0.0
"""mean""",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,1.8243e12,1.7767e12,621468.59636,0.950749
"""std""",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,1.0050e12,1.0455e10,39892.844571,null
"""min""","""deribit""","""btc""","""BTC-10JUL26""","""future""","""inactive""","""BTC""","""BTC""","""BTC""","""USD""","""reversed""","""reversed""","""day""","""call""","""btc_usd""","""1.02e+5""","""1.0""","""0.0001""",1.7828e12,1.5342e12,210838.0,0.0
"""25%""",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,1.7837e12,1.7745e12,606964.0,null
"""50%""",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,1.7879e12,1.7800e12,638395.0,null
"""75%""",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,1.7982e12,1.7824e12,652099.0,null
"""max""","""deribit""","""btc""","""BTC-PERPETUAL""","""option""","""open""","""BTC""","""USD""","""BTC""","""USD""","""reversed""","""reversed""","""week""","""put""","""btc_usd""","""9.8e+4""","""10.0""","""2.5""",3.2504e13,1.7828e12,654669.0,1.0


In [17]:
df.schema

Schema([('exchange', String),
        ('symbol', String),
        ('instrument_name', String),
        ('kind', String),
        ('state', String),
        ('base_currency', String),
        ('quote_currency', String),
        ('settlement_currency', String),
        ('counter_currency', String),
        ('instrument_type', String),
        ('future_type', String),
        ('settlement_period', String),
        ('option_type', String),
        ('price_index', String),
        ('strike', String),
        ('contract_size', String),
        ('tick_size', String),
        ('expiration_timestamp', Int64),
        ('creation_timestamp', Int64),
        ('instrument_id', Int64),
        ('is_active', Boolean)])

## Lazy Stats Helper

`describe_dataset(key)` uses lazy scans and returns count/null/min/max/mean for numeric columns plus count/null/approx unique for string columns. This is safer for the bigger Deribit files than loading every table into memory.

In [18]:
NUMERIC_DTYPES = {
    pl.Int8,
    pl.Int16,
    pl.Int32,
    pl.Int64,
    pl.UInt8,
    pl.UInt16,
    pl.UInt32,
    pl.UInt64,
    pl.Float32,
    pl.Float64,
}


def describe_dataset(key: str) -> pl.DataFrame:
    lf = scan_dataset(key)
    schema = lf.collect_schema()
    exprs = [pl.len().alias("rows")]

    for name, dtype in schema.items():
        col = pl.col(name)
        exprs.append(col.null_count().alias(f"{name}__nulls"))
        if dtype in NUMERIC_DTYPES:
            exprs.extend(
                [
                    col.min().alias(f"{name}__min"),
                    col.max().alias(f"{name}__max"),
                    col.mean().alias(f"{name}__mean"),
                ]
            )
        elif dtype == pl.String:
            exprs.append(col.approx_n_unique().alias(f"{name}__approx_unique"))

    wide = lf.select(exprs).collect()
    rows = []
    for column in wide.columns:
        if column == "rows":
            rows.append({"column": "__table__", "metric": "rows", "value": wide[column][0]})
            continue
        name, metric = column.rsplit("__", 1)
        rows.append({"column": name, "metric": metric, "value": wide[column][0]})
    return pl.DataFrame(rows).sort(["column", "metric"])


describe_dataset(key)

column,metric,value
str,str,f64
"""__table__""","""rows""",934.0
"""base_currency""","""approx_unique""",1.0
"""base_currency""","""nulls""",0.0
"""contract_size""","""approx_unique""",2.0
"""contract_size""","""nulls""",0.0
"""counter_currency""","""approx_unique""",1.0
"""counter_currency""","""nulls""",0.0
"""creation_timestamp""","""max""",1.7828e12
"""creation_timestamp""","""mean""",1.7767e12


## Preview Every Dataset

In [19]:
for dataset_key in sorted(DATASETS):
    print(f"\n{dataset_key}")
    display(load_dataset(dataset_key, limit=5))


bitfinex/tbtcusd/book_l25


exchange,symbol,received_at,received_mts,connection_id,channel,bitfinex_channel_id,event_type,level_index,price,count,amount,amount_abs,side
str,str,str,i64,str,str,i64,str,i64,str,i64,str,str,str
"""bitfinex""","""tBTCUSD""","""2026-06-30T03:24:06.599Z""",1782789846599,"""bitfinex-tBTCUSD-book_l25""","""book_l25""",1779102,"""snapshot""",0,"""59881""",1,"""0.002""","""0.002""","""bid"""
"""bitfinex""","""tBTCUSD""","""2026-06-30T03:24:06.599Z""",1782789846599,"""bitfinex-tBTCUSD-book_l25""","""book_l25""",1779102,"""snapshot""",1,"""59879""",1,"""0.07412797""","""0.07412797""","""bid"""
"""bitfinex""","""tBTCUSD""","""2026-06-30T03:24:06.599Z""",1782789846599,"""bitfinex-tBTCUSD-book_l25""","""book_l25""",1779102,"""snapshot""",2,"""59874""",1,"""0.0835074""","""0.0835074""","""bid"""
"""bitfinex""","""tBTCUSD""","""2026-06-30T03:24:06.599Z""",1782789846599,"""bitfinex-tBTCUSD-book_l25""","""book_l25""",1779102,"""snapshot""",3,"""59873""",1,"""0.08351009""","""0.08351009""","""bid"""
"""bitfinex""","""tBTCUSD""","""2026-06-30T03:24:06.599Z""",1782789846599,"""bitfinex-tBTCUSD-book_l25""","""book_l25""",1779102,"""snapshot""",4,"""59872""",1,"""0.07788546""","""0.07788546""","""bid"""



bitfinex/tbtcusd/trades


exchange,symbol,received_at,received_mts,connection_id,channel,bitfinex_channel_id,event_type,is_final,trade_id,trade_mts,side,amount,amount_abs,price
str,str,str,i64,str,str,i64,str,bool,i64,i64,str,str,str,str
"""bitfinex""","""tBTCUSD""","""2026-06-30T03:24:04.761Z""",1782789844761,"""bitfinex-tBTCUSD-trades""","""trades""",626808,"""snapshot""",true,1942717602,1782789842625,"""sell""","""-0.0002""","""0.0002""","""59896"""
"""bitfinex""","""tBTCUSD""","""2026-06-30T03:24:04.761Z""",1782789844761,"""bitfinex-tBTCUSD-trades""","""trades""",626808,"""snapshot""",true,1942717601,1782789842624,"""sell""","""-0.00011092""","""0.00011092""","""59896"""
"""bitfinex""","""tBTCUSD""","""2026-06-30T03:24:04.761Z""",1782789844761,"""bitfinex-tBTCUSD-trades""","""trades""",626808,"""snapshot""",true,1942717600,1782789842623,"""sell""","""-0.0956""","""0.0956""","""59896"""
"""bitfinex""","""tBTCUSD""","""2026-06-30T03:24:04.761Z""",1782789844761,"""bitfinex-tBTCUSD-trades""","""trades""",626808,"""snapshot""",true,1942717597,1782789842021,"""sell""","""-0.1044""","""0.1044""","""59896"""
"""bitfinex""","""tBTCUSD""","""2026-06-30T03:24:04.761Z""",1782789844761,"""bitfinex-tBTCUSD-trades""","""trades""",626808,"""snapshot""",true,1942717595,1782789841421,"""sell""","""-0.0002""","""0.0002""","""59902"""



deribit/btc/incremental_ticker


...

exchange,symbol,received_at,received_mts,connection_id,channel,coin,book_timestamp,snapshot,side,level_index,price,size,order_count
str,str,str,i64,str,str,str,i64,bool,str,i64,str,str,i64
"""hyperliquid""","""UBTC/USDC""","""2026-06-30T03:24:04.953Z""",1782789844953,"""hyperliquid-spot-btc""","""book""","""@142""",1782789844665,false,"""bid""",0,"""59808.0""","""1.17616""",4
"""hyperliquid""","""UBTC/USDC""","""2026-06-30T03:24:04.953Z""",1782789844953,"""hyperliquid-spot-btc""","""book""","""@142""",1782789844665,false,"""bid""",1,"""59807.0""","""0.18649""",1
"""hyperliquid""","""UBTC/USDC""","""2026-06-30T03:24:04.953Z""",1782789844953,"""hyperliquid-spot-btc""","""book""","""@142""",1782789844665,false,"""bid""",2,"""59805.0""","""0.05201""",1
"""hyperliquid""","""UBTC/USDC""","""2026-06-30T03:24:04.953Z""",1782789844953,"""hyperliquid-spot-btc""","""book""","""@142""",1782789844665,false,"""bid""",3,"""59804.0""","""0.04611""",1
"""hyperliquid""","""UBTC/USDC""","""2026-06-30T03:24:04.953Z""",1782789844953,"""hyperliquid-spot-btc""","""book""","""@142""",1782789844665,false,"""bid""",4,"""59803.0""","""0.18651""",1



hyperliquid/ubtc_usdc/control


exchange,symbol,received_at,received_mts,connection_id,channel,event,message_channel,method,subscription_type,coin,payload_json
str,str,str,i64,str,str,str,str,str,str,str,str
"""hyperliquid""","""UBTC/USDC""","""2026-06-30T03:24:04.818Z""",1782789844818,"""hyperliquid-spot-btc""","""control""","""connected""",null,null,null,null,"""{""event"":""connected""}"""
"""hyperliquid""","""UBTC/USDC""","""2026-06-30T03:24:04.943Z""",1782789844943,"""hyperliquid-spot-btc""","""control""",null,"""subscriptionResponse""","""subscribe""","""trades""","""@142""","""{""channel"":""subscriptionRespon…"
"""hyperliquid""","""UBTC/USDC""","""2026-06-30T03:24:04.953Z""",1782789844953,"""hyperliquid-spot-btc""","""control""",null,"""subscriptionResponse""","""subscribe""","""l2Book""","""@142""","""{""channel"":""subscriptionRespon…"
"""hyperliquid""","""UBTC/USDC""","""2026-06-30T04:52:58.112Z""",1782795178112,"""hyperliquid-spot-btc""","""control""","""connected""",null,null,null,null,"""{""event"":""connected""}"""
"""hyperliquid""","""UBTC/USDC""","""2026-06-30T04:52:58.235Z""",1782795178235,"""hyperliquid-spot-btc""","""control""",null,"""subscriptionResponse""","""subscribe""","""trades""","""@142""","""{""channel"":""subscriptionRespon…"



hyperliquid/ubtc_usdc/trades


exchange,symbol,received_at,received_mts,connection_id,channel,coin,side,raw_side,price,size,trade_timestamp,hash,trade_id,user_0,user_1
str,str,str,i64,str,str,str,str,str,str,str,i64,str,i64,str,str
"""hyperliquid""","""UBTC/USDC""","""2026-06-30T03:24:04.943Z""",1782789844943,"""hyperliquid-spot-btc""","""trades""","""@142""","""buy""","""B""","""59921.0""","""0.00361""",1782789693515,"""0x0f0c30bf6cc618ff1085043ede1c…",656384212498998,"""0xc01b67508210070246baf00adf32…","""0xf9109ada2f73c62e9889b4545306…"
"""hyperliquid""","""UBTC/USDC""","""2026-06-30T03:24:04.943Z""",1782789844943,"""hyperliquid-spot-btc""","""trades""","""@142""","""buy""","""B""","""59917.0""","""0.00035""",1782789697000,"""0x0000000000000000000000000000…",324292006561002,"""0x80a6b9ce006331f2bea01990f731…","""0x1c1c270b573d55b68b3d14722b5d…"
"""hyperliquid""","""UBTC/USDC""","""2026-06-30T03:24:04.943Z""",1782789844943,"""hyperliquid-spot-btc""","""trades""","""@142""","""sell""","""A""","""59914.0""","""0.0002""",1782789702008,"""0x0000000000000000000000000000…",290000434457937,"""0x399965e15d4e61ec3529cc98b7f7…","""0x28f0233472b6a44e170e002a7284…"
"""hyperliquid""","""UBTC/USDC""","""2026-06-30T03:24:04.943Z""",1782789844943,"""hyperliquid-spot-btc""","""trades""","""@142""","""buy""","""B""","""59906.0""","""0.00035""",1782789704016,"""0x0000000000000000000000000000…",191183443988514,"""0x80a6b9ce006331f2bea01990f731…","""0x1c1c270b573d55b68b3d14722b5d…"
"""hyperliquid""","""UBTC/USDC""","""2026-06-30T03:24:04.943Z""",1782789844943,"""hyperliquid-spot-btc""","""trades""","""@142""","""sell""","""A""","""59899.0""","""0.00155""",1782789720000,"""0x0000000000000000000000000000…",979122725840501,"""0x9266865bb6afb4c4f618544dd3b8…","""0xf036a5261406a394bd63eb4df49c…"


## Common Feature Inputs

In [20]:
trade_keys = [key for key in sorted(DATASETS) if key.endswith("/trades")]
book_keys = [key for key in sorted(DATASETS) if key.endswith("/book") or key.endswith("/book_l25") or key.endswith("/orderbook")]
price_keys = [key for key in sorted(DATASETS) if key.endswith("/prices") or key.endswith("/quotes") or key.endswith("/incremental_ticker")]

trade_keys, book_keys, price_keys

(['bitfinex/tbtcusd/trades',
  'deribit/btc/trades',
  'hibachi/btc_usdt-p/trades',
  'hyperliquid/ubtc_usdc/trades'],
 ['bitfinex/tbtcusd/book_l25',
  'hibachi/btc_usdt-p/orderbook',
  'hyperliquid/ubtc_usdc/book'],
 ['deribit/btc/incremental_ticker',
  'hibachi/btc_usdt-p/prices',
  'hibachi/btc_usdt-p/quotes'])

In [21]:
# Canonical Bitfinex trades only. This avoids double-counting te/tu pairs.
if "bitfinex/tbtcusd/trades" in DATASETS:
    bitfinex_final_trades = (
        scan_dataset("bitfinex/tbtcusd/trades")
        .filter(pl.col("is_final"))
        .select(["received_at", "trade_mts", "side", "amount_abs", "price", "event_type", "is_final"])
        .head(20)
        .collect()
    )
    bitfinex_final_trades
else:
    print("bitfinex/tbtcusd/trades not found")

In [22]:
# Quick row-count summary by venue.
summary_by_exchange = (
    pl.from_pandas(summary)
    .group_by("exchange")
    .agg([
        pl.len().alias("datasets"),
        pl.col("rows").sum().alias("rows"),
        pl.col("size_mb").sum().round(3).alias("size_mb"),
    ])
    .sort("exchange")
)
summary_by_exchange

exchange,datasets,rows,size_mb
str,u32,i64,f64
"""bitfinex""",2,162801,1.618
"""deribit""",4,898733,27.474
"""hibachi""",5,205317,1.936
"""hyperliquid""",3,41545,0.231
